# 📱 WhatsApp Outreach Tool — L&D Designs
1. Run Cell 1 — upload your `wigan_hair_leads_*.xlsx` file
2. Run Cell 2 — builds the dashboard
3. Run Cell 3 — downloads it
4. Open the HTML file in your browser and start messaging

In [ ]:
# ── Cell 1: Upload spreadsheet ────────────────────────────────────────────
from google.colab import files
import openpyxl

print('Select your wigan_hair_leads_*.xlsx file...')
uploaded = files.upload()
filename = list(uploaded.keys())[0]

wb = openpyxl.load_workbook(filename)
ws = wb['Leads']
headers = [cell.value for cell in ws[1]]
all_leads = []
for row in ws.iter_rows(min_row=2, values_only=True):
    if row[0]:
        all_leads.append(dict(zip(headers, row)))

print('Loaded', len(all_leads), 'total leads. Filtering...')

In [ ]:
# ── Cell 2: Build dashboard ───────────────────────────────────────────────
import re
from urllib.parse import quote
from datetime import datetime

# ── YOUR MESSAGE ──────────────────────────────────────────────────────────
# Keep it short, friendly, human. {name} is replaced automatically.
MESSAGE = """Hi, I noticed {name} doesn't have a website yet.

I build websites for local barbers and hairdressers in the Wigan area - quick turnaround and affordable.

Free quote if you're interested? - Dylan, L&D Designs"""
# ─────────────────────────────────────────────────────────────────────────

def is_mobile_uk(phone):
    """Return True only if the number is a UK mobile (starts 07)."""
    if not phone:
        return False
    digits = re.sub(r'[^\d]', '', str(phone))
    # Handle +447... or 447... format too
    if digits.startswith('447'):
        return True
    return digits.startswith('07')

def to_wa_number(phone):
    digits = re.sub(r'[^\d]', '', str(phone))
    if digits.startswith('0'):
        return '44' + digits[1:]
    return digits

# ── Filter: no website + UK mobile only ──────────────────────────────────
leads = []
for l in all_leads:
    status = str(l.get('Website Status') or '').upper().strip()
    phone  = str(l.get('Phone Number') or '')
    if status == 'NONE' and is_mobile_uk(phone):
        leads.append(l)

print('Kept', len(leads), 'leads (no website + UK mobile number)')
print('Skipped', len(all_leads) - len(leads), 'leads (has website, landline, or no number)')

# ── Build each card ───────────────────────────────────────────────────────
def make_card(lead, idx):
    name    = str(lead.get('Business Name') or 'Unknown')
    phone   = str(lead.get('Phone Number') or '')
    email   = str(lead.get('Email Address') or '')
    address = str(lead.get('Address') or '')
    dist    = str(lead.get('Distance (miles)') or '')
    card_id = 'c' + str(idx)

    msg     = MESSAGE.format(name=name)
    wa_num  = to_wa_number(phone)
    wa_link = 'https://wa.me/' + wa_num + '?text=' + quote(msg)
    maps_q  = quote(name + ' ' + address)
    maps_link = 'https://www.google.com/maps/search/' + maps_q

    email_line = ('<div class="det">&#9993; ' + email + '</div>') if email else ''
    dist_txt   = (dist + ' mi') if dist else ''

    return (
        '<div class="card" id="' + card_id + '">'
        '<div class="card-top">'
        '<div><div class="bname">' + name + '</div>'
        '<div class="addr">' + address + '</div></div>'
        '<div class="dist">' + dist_txt + '</div>'
        '</div>'
        '<div class="det">&#128222; ' + phone + '</div>'
        + email_line +
        '<div class="status-row">'
        '<select class="status-sel" onchange="saveStatus(\''+card_id+'\')">'
        '<option value="new">&#128310; Not contacted</option>'
        '<option value="sent">&#128226; Messaged</option>'
        '<option value="replied">&#128172; Replied</option>'
        '<option value="interested">&#9989; Interested</option>'
        '<option value="booked">&#127881; Booked!</option>'
        '<option value="no">&#10060; Not interested</option>'
        '</select>'
        '</div>'
        '<textarea class="notes" placeholder="Notes..." oninput="saveNotes(\''+card_id+'\')" rows="2"></textarea>'
        '<div class="btns">'
        '<a class="wa" href="' + wa_link + '" target="_blank" onclick="setSent(\''+card_id+'\')">&#128248; WhatsApp</a>'
        '<a class="call" href="tel:' + phone + '">&#128222; Call</a>'
        '<a class="maps" href="' + maps_link + '" target="_blank">&#128205; Maps</a>'
        '</div>'
        '</div>'
    )

cards_html = '\n'.join(make_card(l, i) for i, l in enumerate(leads))
total = len(leads)

# ── HTML ──────────────────────────────────────────────────────────────────
html = '<!DOCTYPE html><html lang="en"><head><meta charset="UTF-8">'
html += '<meta name="viewport" content="width=device-width,initial-scale=1">'
html += '<title>L&D Designs Outreach</title><style>'
html += '''
* { box-sizing:border-box; margin:0; padding:0; }
body { font-family:-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif; background:#f0f2f5; }
.header { background:#1a2035; color:white; padding:16px 20px; }
.header h1 { font-size:1.2rem; }
.header p  { opacity:.6; font-size:.8rem; margin-top:2px; }
.topbar { display:flex; gap:8px; flex-wrap:wrap; align-items:center;
           padding:12px 18px; background:white; border-bottom:1px solid #e0e0e0; }
.fbtn { padding:5px 13px; border:2px solid #ddd; border-radius:16px;
         background:white; cursor:pointer; font-size:.78rem; font-weight:500; }
.fbtn.on { border-color:#1a2035; background:#1a2035; color:white; }
.progress { margin-left:auto; font-size:.8rem; color:#666; }
.progress strong { color:#1a2035; }
.stats { display:flex; gap:8px; flex-wrap:wrap; padding:10px 18px;
          background:white; border-bottom:1px solid #e8e8e8; }
.stat { background:#f7f7f7; border-radius:6px; padding:8px 14px; text-align:center; }
.stat .n { font-size:1.4rem; font-weight:700; color:#1a2035; line-height:1; }
.stat .l { font-size:.68rem; color:#aaa; margin-top:2px; }
.grid { display:grid; grid-template-columns:repeat(auto-fill,minmax(300px,1fr));
         gap:10px; padding:14px 18px; }
.card { background:white; border-radius:8px; padding:14px;
         box-shadow:0 1px 3px rgba(0,0,0,.06); border-left:3px solid #e74c3c;
         transition:opacity .3s, border-color .3s; }
.card.status-sent      { border-color:#3498db; }
.card.status-replied   { border-color:#9b59b6; }
.card.status-interested{ border-color:#27ae60; }
.card.status-booked    { border-color:#f39c12; opacity:.7; }
.card.status-no        { opacity:.3; }
.card-top { display:flex; justify-content:space-between; gap:8px; margin-bottom:8px; }
.bname { font-weight:700; font-size:.92rem; color:#1a2035; }
.addr  { font-size:.73rem; color:#aaa; margin-top:2px; }
.dist  { font-size:.75rem; color:#bbb; white-space:nowrap; }
.det   { font-size:.78rem; color:#666; margin:2px 0; }
.status-row { margin:10px 0 6px; }
.status-sel { width:100%; padding:6px 8px; border:1px solid #e0e0e0;
               border-radius:5px; font-size:.8rem; background:#fafafa; cursor:pointer; }
.notes { width:100%; border:1px solid #e8e8e8; border-radius:5px;
          padding:6px 8px; font-size:.78rem; color:#555; resize:vertical;
          font-family:inherit; background:#fafafa; margin-bottom:8px; }
.notes:focus { outline:none; border-color:#1a2035; }
.btns { display:flex; gap:6px; flex-wrap:wrap; }
.wa   { background:#25D366; color:white; padding:7px 14px; border-radius:5px;
         text-decoration:none; font-size:.78rem; font-weight:700; }
.call { background:#1a2035; color:white; padding:7px 12px; border-radius:5px;
         text-decoration:none; font-size:.78rem; font-weight:600; }
.maps { background:#f0f2f5; color:#555; padding:7px 12px; border-radius:5px;
         text-decoration:none; font-size:.78rem; border:1px solid #ddd; }
.wa:hover { background:#1ebe5d; } .call:hover { background:#2c3e6b; }
'''
html += '</style></head><body>'

html += '<div class="header"><h1>&#128248; L&amp;D Designs &mdash; Outreach Dashboard</h1>'
html += '<p>No-website leads with UK mobile numbers &nbsp;&middot;&nbsp; '
html += datetime.now().strftime('%d/%m/%Y') + ' &nbsp;&middot;&nbsp; Progress saves automatically</p></div>'

html += '<div class="stats">'
html += '<div class="stat"><div class="n">' + str(total) + '</div><div class="l">Leads</div></div>'
html += '<div class="stat"><div class="n" id="s-new">' + str(total) + '</div><div class="l">To Contact</div></div>'
html += '<div class="stat"><div class="n" id="s-sent">0</div><div class="l">Messaged</div></div>'
html += '<div class="stat"><div class="n" id="s-interested">0</div><div class="l">Interested</div></div>'
html += '<div class="stat"><div class="n" id="s-booked">0</div><div class="l">Booked</div></div>'
html += '</div>'

html += '<div class="topbar">'
html += '<button class="fbtn on" onclick="filt(this,\'all\')">All</button>'
html += '<button class="fbtn" onclick="filt(this,\'new\')">Not contacted</button>'
html += '<button class="fbtn" onclick="filt(this,\'sent\')">Messaged</button>'
html += '<button class="fbtn" onclick="filt(this,\'replied\')">Replied</button>'
html += '<button class="fbtn" onclick="filt(this,\'interested\')">Interested</button>'
html += '<button class="fbtn" onclick="filt(this,\'booked\')">Booked</button>'
html += '</div>'

html += '<div class="grid" id="grid">' + cards_html + '</div>'

html += '''
<script>
var STORE_KEY = 'ld_outreach_v1';

function getData() {
  try { return JSON.parse(localStorage.getItem(STORE_KEY)) || {}; } catch(e) { return {}; }
}
function saveData(d) { localStorage.setItem(STORE_KEY, JSON.stringify(d)); }

function saveStatus(id) {
  var sel = document.querySelector('#'+id+' .status-sel');
  var val = sel.value;
  var d = getData(); if (!d[id]) d[id] = {};
  d[id].status = val; saveData(d);
  var card = document.getElementById(id);
  card.className = 'card status-' + val;
  updateStats();
}

function saveNotes(id) {
  var ta = document.querySelector('#'+id+' .notes');
  var d = getData(); if (!d[id]) d[id] = {};
  d[id].notes = ta.value; saveData(d);
}

function setSent(id) {
  setTimeout(function() {
    var sel = document.querySelector('#'+id+' .status-sel');
    if (sel && sel.value === 'new') {
      sel.value = 'sent';
      saveStatus(id);
    }
  }, 2000);
}

function updateStats() {
  var counts = {new:0, sent:0, replied:0, interested:0, booked:0, no:0};
  document.querySelectorAll('.status-sel').forEach(function(s) {
    if (counts[s.value] !== undefined) counts[s.value]++;
  });
  var el;
  el = document.getElementById('s-new');       if(el) el.textContent = counts.new;
  el = document.getElementById('s-sent');      if(el) el.textContent = counts.sent + counts.replied;
  el = document.getElementById('s-interested');if(el) el.textContent = counts.interested;
  el = document.getElementById('s-booked');    if(el) el.textContent = counts.booked;
}

function filt(btn, type) {
  document.querySelectorAll('.fbtn').forEach(function(b){ b.classList.remove('on'); });
  btn.classList.add('on');
  document.querySelectorAll('.card').forEach(function(card) {
    var sel = card.querySelector('.status-sel');
    var val = sel ? sel.value : 'new';
    card.style.display = (type === 'all' || type === val) ? '' : 'none';
  });
}

// Restore saved state on load
window.addEventListener('load', function() {
  var d = getData();
  Object.keys(d).forEach(function(id) {
    var card = document.getElementById(id);
    if (!card) return;
    if (d[id].status) {
      var sel = card.querySelector('.status-sel');
      if (sel) { sel.value = d[id].status; card.className = 'card status-' + d[id].status; }
    }
    if (d[id].notes) {
      var ta = card.querySelector('.notes');
      if (ta) ta.value = d[id].notes;
    }
  });
  updateStats();
});
</script></body></html>'''

with open('outreach_dashboard.html', 'w', encoding='utf-8') as f:
    f.write(html)

print('Done!', total, 'leads in your dashboard.')

In [ ]:
# ── Cell 3: Download ──────────────────────────────────────────────────────
from google.colab import files
files.download('outreach_dashboard.html')
print('Check your Downloads folder for outreach_dashboard.html')